# AI Workforce Capacity Planning Platform

## Notebook 00 — Enterprise Project Bootstrap

**Implementation:** 28 — Enterprise Release Remediation  
**Platform release:** v3.0.0  
**Notebook role:** Shared Databricks Runtime Bootstrap  
**Canonical Python namespace:** `src.*`

---

### Purpose

This notebook provides the shared runtime bootstrap for the active Databricks
notebooks in the AI Workforce Capacity Planning Platform.

It is responsible for:

1. locating the repository root,
2. establishing the canonical `src.*` Python namespace,
3. defining shared runtime and persistent-storage configuration,
4. validating the Databricks execution environment,
5. exposing a minimal runtime contract to downstream notebooks.

### Architecture Contract

Reusable platform implementation belongs in `src/`.

This notebook must remain a thin runtime/bootstrap layer and must not
reimplement forecasting, validation, metadata, workforce-planning,
optimization, orchestration, reporting, monitoring, or API business logic.

### Release Finding

**ENG-001 — Inconsistent Python import namespaces**

All active notebooks must use the canonical platform namespace:

`src.*`

Legacy imports such as `forecast.*`, `planning.*`, `monitoring.*`,
or other unprefixed platform-package namespaces are not part of the
v3.0.0 release contract.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Remediation
#
# Notebook:
#     00_project_setup
#
# Section:
#     Canonical Repository and Python Runtime Bootstrap
#
# Platform Release:
#     v3.0.0
#
# Release Finding:
#     ENG-001 — Inconsistent Python Import Namespaces
# =============================================================================

from __future__ import annotations

import importlib
import sys
from pathlib import Path


# -----------------------------------------------------------------------------
# Repository discovery
# -----------------------------------------------------------------------------

def _locate_repository_root() -> Path:
    """
    Locate the AI Workforce Capacity Planning Platform repository root.

    The repository root is identified structurally rather than through a
    user-specific Databricks workspace path.
    """

    start_path = Path.cwd().resolve()

    candidates = (
        start_path,
        *start_path.parents,
    )

    for candidate in candidates:
        src_directory = candidate / "src"
        notebook_directory = candidate / "notebooks" / "source"

        if (
            src_directory.is_dir()
            and notebook_directory.is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Unable to locate the AI Workforce Capacity Planning Platform "
        "repository root from the current Databricks working directory."
    )


REPOSITORY_ROOT = _locate_repository_root()
SRC_ROOT = REPOSITORY_ROOT / "src"


# -----------------------------------------------------------------------------
# Canonical Python namespace registration
# -----------------------------------------------------------------------------
#
# Canonical platform imports use:
#
#     from src.<package> ...
#
# Therefore the repository ROOT belongs on sys.path.
#
# We intentionally do not require legacy imports such as:
#
#     from forecast ...
#     from planning ...
#     from monitoring ...
#
# -----------------------------------------------------------------------------

repository_root_string = str(REPOSITORY_ROOT)

if repository_root_string not in sys.path:
    sys.path.insert(
        0,
        repository_root_string,
    )


# -----------------------------------------------------------------------------
# Canonical namespace validation
# -----------------------------------------------------------------------------

src_package = importlib.import_module("src")

assert src_package.__name__ == "src"

# ``src`` may be loaded as a regular package or as a PEP 420 namespace
# package. Namespace packages do not necessarily define ``__file__``.
# Validate the package through its search locations instead.

src_search_locations = tuple(
    Path(location).resolve()
    for location in getattr(
        src_package.__spec__,
        "submodule_search_locations",
        (),
    )
)

assert src_search_locations, (
    "The canonical src package does not expose any "
    "package search locations."
)

assert SRC_ROOT.resolve() in src_search_locations, (
    "The imported src namespace does not include the active "
    "repository source directory.\n"
    f"Expected: {SRC_ROOT.resolve()}\n"
    f"Actual:   {src_search_locations}"
)


# -----------------------------------------------------------------------------
# Reject already-loaded legacy platform namespaces
# -----------------------------------------------------------------------------

LEGACY_PLATFORM_NAMESPACES = (
    "api",
    "application",
    "bootstrap",
    "demand",
    "forecast",
    "metadata",
    "monitoring",
    "optimization",
    "orchestration",
    "overtime",
    "planning",
    "reporting",
    "runner",
    "staffing",
    "validation",
    "workforce",
)

loaded_module_names = tuple(sys.modules.keys())

loaded_legacy_modules = sorted(
    module_name
    for module_name in loaded_module_names
    if any(
        module_name == namespace
        or module_name.startswith(
            f"{namespace}."
        )
        for namespace in LEGACY_PLATFORM_NAMESPACES
    )
)

assert loaded_legacy_modules == [], (
    "Legacy platform namespaces are already loaded in this "
    "Python session. Restart Python before continuing.\n"
    f"Detected modules: {loaded_legacy_modules}"
)


# -----------------------------------------------------------------------------
# Runtime contract
# -----------------------------------------------------------------------------

PLATFORM_RELEASE = "v3.0.0"
CANONICAL_NAMESPACE = "src.*"
RELEASE_FINDING = "ENG-001"

print("=" * 80)
print("AI WORKFORCE CAPACITY PLANNING PLATFORM")
print("ENTERPRISE PROJECT BOOTSTRAP")
print("=" * 80)
print(f"Repository root:     {REPOSITORY_ROOT}")
print(f"Source root:         {SRC_ROOT}")
print(f"Platform release:    {PLATFORM_RELEASE}")
print(f"Canonical namespace: {CANONICAL_NAMESPACE}")
print(f"Release finding:     {RELEASE_FINDING}")
print("Runtime bootstrap:   PASSED")
print("=" * 80)

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Remediation
#
# Notebook:
#     00_project_setup
#
# Section:
#     Shared Persistent Storage Configuration
#
# Platform Release:
#     v3.0.0
# =============================================================================

from dataclasses import dataclass


@dataclass(frozen=True)
class PlatformStorageConfiguration:
    """
    Persistent storage contract shared by active platform notebooks.

    Paths represent durable platform assets rather than notebook-local
    temporary storage.
    """

    s3_bucket: str
    s3_prefix: str

    @property
    def s3_root(self) -> str:
        return f"s3a://{self.s3_bucket}/{self.s3_prefix}"

    @property
    def landing_path(self) -> str:
        return f"{self.s3_root}/landing"

    @property
    def bronze_path(self) -> str:
        return f"{self.s3_root}/bronze"

    @property
    def silver_path(self) -> str:
        return f"{self.s3_root}/silver"

    @property
    def gold_path(self) -> str:
        return f"{self.s3_root}/gold"

    @property
    def registry_path(self) -> str:
        return f"{self.s3_root}/shared/registry"

    @property
    def metadata_path(self) -> str:
        return f"{self.s3_root}/shared/metadata"

    @property
    def models_path(self) -> str:
        return f"{self.s3_root}/shared/models"

    @property
    def experiments_path(self) -> str:
        return f"{self.s3_root}/shared/experiments"

    @property
    def reports_path(self) -> str:
        return f"{self.s3_root}/shared/reports"

    @property
    def checkpoints_path(self) -> str:
        return f"{self.s3_root}/shared/checkpoints"


STORAGE = PlatformStorageConfiguration(
    s3_bucket="issouf-data-lake",
    s3_prefix="overtime-capacity-planning",
)

# -----------------------------------------------------------------------------
# Backward-compatible notebook aliases
# -----------------------------------------------------------------------------
#
# These aliases temporarily preserve the runtime contract expected by the
# existing active notebooks while those notebooks are remediated individually.
# They are configuration aliases only — not duplicate business logic.
# -----------------------------------------------------------------------------

S3_BUCKET = STORAGE.s3_bucket
S3_PREFIX = STORAGE.s3_prefix
S3_ROOT = STORAGE.s3_root

LANDING_PATH = STORAGE.landing_path
BRONZE_PATH = STORAGE.bronze_path
SILVER_PATH = STORAGE.silver_path
GOLD_PATH = STORAGE.gold_path

REGISTRY_PATH = STORAGE.registry_path
METADATA_PATH = STORAGE.metadata_path
MODELS_PATH = STORAGE.models_path
EXPERIMENTS_PATH = STORAGE.experiments_path
REPORTS_PATH = STORAGE.reports_path
CHECKPOINTS_PATH = STORAGE.checkpoints_path


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert STORAGE.s3_root.startswith("s3a://")
assert LANDING_PATH.endswith("/landing")
assert BRONZE_PATH.endswith("/bronze")
assert SILVER_PATH.endswith("/silver")
assert GOLD_PATH.endswith("/gold")

persistent_paths = (
    LANDING_PATH,
    BRONZE_PATH,
    SILVER_PATH,
    GOLD_PATH,
    REGISTRY_PATH,
    METADATA_PATH,
    MODELS_PATH,
    EXPERIMENTS_PATH,
    REPORTS_PATH,
    CHECKPOINTS_PATH,
)

assert all(
    path.startswith(STORAGE.s3_root)
    for path in persistent_paths
)

print("=" * 80)
print("SHARED PERSISTENT STORAGE CONFIGURATION")
print("=" * 80)
print(f"S3 root:        {S3_ROOT}")
print(f"Landing:        {LANDING_PATH}")
print(f"Bronze:         {BRONZE_PATH}")
print(f"Silver:         {SILVER_PATH}")
print(f"Gold:           {GOLD_PATH}")
print(f"Registry:       {REGISTRY_PATH}")
print(f"Metadata:       {METADATA_PATH}")
print(f"Models:         {MODELS_PATH}")
print(f"Experiments:    {EXPERIMENTS_PATH}")
print(f"Reports:        {REPORTS_PATH}")
print(f"Checkpoints:    {CHECKPOINTS_PATH}")
print("Storage config: PASSED")
print("=" * 80)

## Section 01 — Project Identity

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 01:
#     Project Identity
#
# Release:
#     v3.0.0
#
# Finding:
#     ENG-001
# =============================================================================

from datetime import datetime, timezone
from typing import Any


# -----------------------------------------------------------------------------
# Project identity
# -----------------------------------------------------------------------------

PROJECT_NAME = "AI Workforce Capacity Planning Platform"
PROJECT_KEY = "overtime-capacity-planning"

PROJECT_VERSION = "3.0.0"
PROJECT_SETUP_VERSION = "3.0.0"

ENVIRONMENT = "development"
PROJECT_OWNER = "Issouf KABRE"

PROJECT_INITIALIZED_AT_UTC = datetime.now(timezone.utc)


# -----------------------------------------------------------------------------
# Canonical project configuration
# -----------------------------------------------------------------------------

PROJECT_CONFIG: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "project_key": PROJECT_KEY,
    "project_version": PROJECT_VERSION,
    "project_setup_version": PROJECT_SETUP_VERSION,
    "environment": ENVIRONMENT,
    "project_owner": PROJECT_OWNER,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert PROJECT_NAME == "AI Workforce Capacity Planning Platform"
assert PROJECT_KEY == "overtime-capacity-planning"

# PLATFORM_RELEASE is the canonical release label: "v3.0.0".
# Project versions use the semantic version form: "3.0.0".
assert f"v{PROJECT_VERSION}" == PLATFORM_RELEASE
assert f"v{PROJECT_SETUP_VERSION}" == PLATFORM_RELEASE

assert ENVIRONMENT in {
    "development",
    "test",
    "staging",
    "production",
}

assert (
    f"v{PROJECT_CONFIG['project_version']}"
    == PLATFORM_RELEASE
)

assert (
    f"v{PROJECT_CONFIG['project_setup_version']}"
    == PLATFORM_RELEASE
)


print("=" * 80)
print("PROJECT IDENTITY")
print("=" * 80)
print(f"Project name:          {PROJECT_NAME}")
print(f"Project key:           {PROJECT_KEY}")
print(f"Project version:       {PROJECT_VERSION}")
print(f"Project setup version: {PROJECT_SETUP_VERSION}")
print(f"Environment:           {ENVIRONMENT}")
print(f"Project owner:         {PROJECT_OWNER}")
print(f"Initialized at UTC:    {PROJECT_INITIALIZED_AT_UTC.isoformat()}")
print("Project identity:      PASSED")
print("=" * 80)

## Section 02 — Persistent Storage Configuration

The platform uses Amazon S3 for durable analytical and operational artifacts.

The canonical storage root is established by the shared
`PlatformStorageConfiguration` contract initialized during bootstrap.

Bronze remains the first persisted analytical data layer. A `landing`
namespace is retained for acquisition and staging workflows, but the current
pipeline does not require every source file to be permanently duplicated there.

This section derives all downstream notebook paths from the canonical storage
configuration and does not redefine the S3 bucket or project root.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 02:
#     Persistent Storage Configuration
#
# Release:
#     v3.0.0
# =============================================================================


# -----------------------------------------------------------------------------
# Canonical roots
# -----------------------------------------------------------------------------
#
# STORAGE, S3_ROOT, LANDING_PATH, BRONZE_PATH, SILVER_PATH, and GOLD_PATH
# were established by the shared storage bootstrap.
#
# Do not redefine the S3 bucket here.
# -----------------------------------------------------------------------------

PROJECT_ROOT = S3_ROOT

LANDING_ROOT = LANDING_PATH
BRONZE_ROOT = BRONZE_PATH
SILVER_ROOT = SILVER_PATH
GOLD_ROOT = GOLD_PATH


# -----------------------------------------------------------------------------
# Shared platform artifact roots
# -----------------------------------------------------------------------------

METADATA_ROOT = METADATA_PATH
REGISTRY_ROOT = REGISTRY_PATH

MANIFEST_ROOT = f"{METADATA_ROOT}/manifests"
VALIDATION_ROOT = f"{METADATA_ROOT}/validation"
PIPELINE_LOG_ROOT = f"{METADATA_ROOT}/pipeline_logs"

MODEL_ROOT = MODELS_PATH
FORECAST_ROOT = f"{PROJECT_ROOT}/forecasts"
DECISION_ROOT = f"{PROJECT_ROOT}/decisions"
REPORT_ROOT = REPORTS_PATH

DATASET_REGISTRY_PATH = (
    f"{REGISTRY_ROOT}/datasets"
)


# -----------------------------------------------------------------------------
# Structured storage configuration
# -----------------------------------------------------------------------------

STORAGE_CONFIG: dict[str, str] = {
    "s3_bucket": S3_BUCKET,
    "project_root": PROJECT_ROOT,
    "landing_root": LANDING_ROOT,
    "bronze_root": BRONZE_ROOT,
    "silver_root": SILVER_ROOT,
    "gold_root": GOLD_ROOT,
    "metadata_root": METADATA_ROOT,
    "registry_root": REGISTRY_ROOT,
    "dataset_registry_path": DATASET_REGISTRY_PATH,
    "manifest_root": MANIFEST_ROOT,
    "validation_root": VALIDATION_ROOT,
    "pipeline_log_root": PIPELINE_LOG_ROOT,
    "model_root": MODEL_ROOT,
    "forecast_root": FORECAST_ROOT,
    "decision_root": DECISION_ROOT,
    "report_root": REPORT_ROOT,
}


# -----------------------------------------------------------------------------
# Temporary compatibility map for downstream notebooks
# -----------------------------------------------------------------------------
#
# 01_dataset_evaluation and 02_data_pipeline still consume several of these
# names. They remain aliases to the canonical configuration while those
# notebooks are remediated individually.
# -----------------------------------------------------------------------------

PROJECT_PATHS: dict[str, str] = {
    "project_root": PROJECT_ROOT,
    "landing": LANDING_ROOT,
    "bronze": BRONZE_ROOT,
    "silver": SILVER_ROOT,
    "gold": GOLD_ROOT,
    "metadata": METADATA_ROOT,
    "registry": REGISTRY_ROOT,
    "dataset_registry": DATASET_REGISTRY_PATH,
    "manifests": MANIFEST_ROOT,
    "validation": VALIDATION_ROOT,
    "pipeline_logs": PIPELINE_LOG_ROOT,
    "models": MODEL_ROOT,
    "forecasts": FORECAST_ROOT,
    "decisions": DECISION_ROOT,
    "reports": REPORT_ROOT,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert PROJECT_ROOT == STORAGE.s3_root

assert LANDING_ROOT == STORAGE.landing_path
assert BRONZE_ROOT == STORAGE.bronze_path
assert SILVER_ROOT == STORAGE.silver_path
assert GOLD_ROOT == STORAGE.gold_path

assert METADATA_ROOT == STORAGE.metadata_path
assert REGISTRY_ROOT == STORAGE.registry_path
assert MODEL_ROOT == STORAGE.models_path
assert REPORT_ROOT == STORAGE.reports_path

assert all(
    path.startswith(PROJECT_ROOT)
    for path in (
        LANDING_ROOT,
        BRONZE_ROOT,
        SILVER_ROOT,
        GOLD_ROOT,
        METADATA_ROOT,
        REGISTRY_ROOT,
        MODEL_ROOT,
        FORECAST_ROOT,
        DECISION_ROOT,
        REPORT_ROOT,
    )
)


print("=" * 80)
print("PERSISTENT STORAGE CONFIGURATION")
print("=" * 80)
print(f"Project root:       {PROJECT_ROOT}")
print(f"Landing root:       {LANDING_ROOT}")
print(f"Bronze root:        {BRONZE_ROOT}")
print(f"Silver root:        {SILVER_ROOT}")
print(f"Gold root:          {GOLD_ROOT}")
print(f"Metadata root:      {METADATA_ROOT}")
print(f"Registry root:      {REGISTRY_ROOT}")
print(f"Model root:         {MODEL_ROOT}")
print(f"Forecast root:      {FORECAST_ROOT}")
print(f"Decision root:      {DECISION_ROOT}")
print(f"Report root:        {REPORT_ROOT}")
print("Storage contract:   PASSED")
print("=" * 80)

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 02.1:
#     Governed Unity Catalog Storage
#
# Release:
#     v3.0.0
# =============================================================================


# -----------------------------------------------------------------------------
# Governed metadata storage
# -----------------------------------------------------------------------------
#
# Durable analytical datasets and operational platform artifacts remain in S3.
#
# Unity Catalog Volumes provide governed workspace storage for metadata catalog
# persistence, governance artifacts, and controlled demonstration datasets.
#
# This is intentionally separate from:
#
#     METADATA_ROOT = s3://.../shared/metadata
#
# The two locations serve different storage responsibilities.
# -----------------------------------------------------------------------------

UNITY_CATALOG_VOLUME_ROOT = (
    "/Volumes/dev/default/project_storage"
)

GOVERNED_METADATA_ROOT = (
    f"{UNITY_CATALOG_VOLUME_ROOT}/enterprise_metadata"
)

GOVERNED_METADATA_CATALOG_PATH = (
    f"{GOVERNED_METADATA_ROOT}/catalog"
)

GOVERNED_SAMPLE_DATASET_PATH = (
    f"{GOVERNED_METADATA_ROOT}/sample_dataset"
)


UNITY_CATALOG_STORAGE_CONFIG: dict[str, str] = {
    "volume_root": UNITY_CATALOG_VOLUME_ROOT,
    "governed_metadata_root": GOVERNED_METADATA_ROOT,
    "metadata_catalog_path": GOVERNED_METADATA_CATALOG_PATH,
    "sample_dataset_path": GOVERNED_SAMPLE_DATASET_PATH,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert UNITY_CATALOG_VOLUME_ROOT.startswith(
    "/Volumes/"
)

assert GOVERNED_METADATA_ROOT.startswith(
    UNITY_CATALOG_VOLUME_ROOT
)

assert GOVERNED_METADATA_CATALOG_PATH.startswith(
    GOVERNED_METADATA_ROOT
)

assert GOVERNED_SAMPLE_DATASET_PATH.startswith(
    GOVERNED_METADATA_ROOT
)

# S3 operational metadata and governed UC metadata must remain distinct.
assert GOVERNED_METADATA_ROOT != METADATA_ROOT
assert METADATA_ROOT.startswith("s3a://")


print("=" * 80)
print("GOVERNED UNITY CATALOG STORAGE")
print("=" * 80)
print(
    f"Volume root:              "
    f"{UNITY_CATALOG_VOLUME_ROOT}"
)
print(
    f"Governed metadata root:   "
    f"{GOVERNED_METADATA_ROOT}"
)
print(
    f"Metadata catalog path:    "
    f"{GOVERNED_METADATA_CATALOG_PATH}"
)
print(
    f"Sample dataset path:      "
    f"{GOVERNED_SAMPLE_DATASET_PATH}"
)
print(
    f"S3 operational metadata: "
    f"{METADATA_ROOT}"
)
print("Governed storage:         PASSED")
print("=" * 80)

### 2.1 Governed Unity Catalog Storage

Unity Catalog Volumes provide governed storage for metadata catalog artifacts
and controlled demonstration datasets.

Persistent analytical datasets and operational platform artifacts continue to
use the canonical S3 hierarchy.

The two storage systems are intentionally complementary:

- **Amazon S3** — analytical datasets and durable platform artifacts
- **Unity Catalog Volumes** — governed metadata/catalog and demonstration assets

This separation avoids multiple competing definitions of the platform's
canonical S3 metadata root.

## Section 03 — Data-Pipeline Parameters

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 03:
#     Data-Pipeline Parameters
#
# Release:
#     v3.0.0
# =============================================================================

PIPELINE_CONFIG: dict[str, Any] = {
    "pipeline_name": "enterprise-workforce-data-foundation",
    "pipeline_version": PLATFORM_RELEASE,
    "write_mode": "overwrite",
    "parquet_compression": "snappy",
    "enable_data_quality_checks": True,
    "save_execution_log": True,
    "fail_on_empty_dataset": True,
    "fail_on_row_count_mismatch": True,
    "fail_on_duplicate_business_keys": True,
}


# -----------------------------------------------------------------------------
# Backward-compatible constants for downstream notebooks
# -----------------------------------------------------------------------------

PIPELINE_NAME = PIPELINE_CONFIG["pipeline_name"]
PIPELINE_VERSION = PIPELINE_CONFIG["pipeline_version"]
PARQUET_WRITE_MODE = PIPELINE_CONFIG["write_mode"]
PARQUET_COMPRESSION = PIPELINE_CONFIG["parquet_compression"]


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert PIPELINE_NAME == "enterprise-workforce-data-foundation"
assert PIPELINE_VERSION == PLATFORM_RELEASE

assert PARQUET_WRITE_MODE in {
    "overwrite",
    "append",
}

assert PARQUET_COMPRESSION in {
    "snappy",
    "gzip",
    "zstd",
}

assert PIPELINE_CONFIG["enable_data_quality_checks"] is True
assert PIPELINE_CONFIG["save_execution_log"] is True
assert PIPELINE_CONFIG["fail_on_empty_dataset"] is True
assert PIPELINE_CONFIG["fail_on_row_count_mismatch"] is True
assert PIPELINE_CONFIG["fail_on_duplicate_business_keys"] is True


print("=" * 80)
print("DATA-PIPELINE PARAMETERS")
print("=" * 80)
print(f"Pipeline name:       {PIPELINE_NAME}")
print(f"Pipeline version:    {PIPELINE_VERSION}")
print(f"Write mode:          {PARQUET_WRITE_MODE}")
print(f"Compression:         {PARQUET_COMPRESSION}")
print(
    "Data quality checks: "
    f"{PIPELINE_CONFIG['enable_data_quality_checks']}"
)
print(
    "Execution logging:   "
    f"{PIPELINE_CONFIG['save_execution_log']}"
)
print("Pipeline contract:   PASSED")
print("=" * 80)

## Section 04 — Forecast Parameters

The active forecast horizon is a runtime parameter.

This configuration defines:

- a default value,
- accepted boundaries,
- validation behavior,
- forecasting metadata.

It does **not** hard-code the business to one-day, seven-day, or
fourteen-day forecasting.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 21 — Enterprise Platform Integration
# Notebook: 00_project_setup
# Section 04 — Forecast Parameters
# =============================================================================

FORECAST_CONFIG: dict[str, Any] = {
    "default_horizon_days": 14,
    "supported_horizon_days": (1, 7, 14, 30, 60, 90),
    "frequency": "daily",
    "date_column": "order_date",
    "target_column": "order_line_count",
    "validation_horizon_days": 28,
    "minimum_training_rows": 180,
    "random_seed": 42,
    "retrain_model": True,
    "save_model": True,
    "confidence_level": 0.95,
}

DEFAULT_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["default_horizon_days"]
SUPPORTED_FORECAST_HORIZON_DAYS = FORECAST_CONFIG["supported_horizon_days"]

# Backward-compatible boundaries for downstream notebook consumers.
MINIMUM_FORECAST_HORIZON_DAYS = min(SUPPORTED_FORECAST_HORIZON_DAYS)
MAXIMUM_FORECAST_HORIZON_DAYS = max(SUPPORTED_FORECAST_HORIZON_DAYS)

# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert DEFAULT_FORECAST_HORIZON_DAYS in SUPPORTED_FORECAST_HORIZON_DAYS

assert SUPPORTED_FORECAST_HORIZON_DAYS == (
    1,
    7,
    14,
    30,
    60,
    90,
)

assert MINIMUM_FORECAST_HORIZON_DAYS == 1
assert MAXIMUM_FORECAST_HORIZON_DAYS == 90

assert FORECAST_CONFIG["frequency"] == "daily"
assert FORECAST_CONFIG["date_column"] == "order_date"
assert FORECAST_CONFIG["target_column"] == "order_line_count"

assert FORECAST_CONFIG["validation_horizon_days"] > 0
assert FORECAST_CONFIG["minimum_training_rows"] > 0

assert 0.0 < FORECAST_CONFIG["confidence_level"] < 1.0

print("=" * 80)
print("FORECAST PARAMETERS")
print("=" * 80)
print(f"Default horizon:       {DEFAULT_FORECAST_HORIZON_DAYS} days")
print(f"Supported horizons:    {SUPPORTED_FORECAST_HORIZON_DAYS}")
print(f"Frequency:             {FORECAST_CONFIG['frequency']}")
print(f"Date column:           {FORECAST_CONFIG['date_column']}")
print(f"Primary target:        {FORECAST_CONFIG['target_column']}")
print(f"Validation horizon:    {FORECAST_CONFIG['validation_horizon_days']} days")
print(f"Minimum training rows: {FORECAST_CONFIG['minimum_training_rows']}")
print(f"Confidence level:      {FORECAST_CONFIG['confidence_level']}")
print("Forecast contract:     PASSED")
print("=" * 80)

## Section 05 — Model Parameters

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 05:
#     Model Runtime Parameters
#
# Release:
#     v3.0.0
# =============================================================================


MODEL_CONFIG: dict[str, Any] = {
    "enabled_models": (
        "naive_last_value",
        "linear_regression",
        "random_forest",
    ),
    "primary_metric": "wape",
    "secondary_metrics": (
        "mae",
        "rmse",
        "mape",
        "smape",
        "bias",
    ),
    "random_seed": 42,
    "training_ratio": 0.70,
    "validation_ratio": 0.15,
    "test_ratio": 0.15,
    "shuffle_training": False,
    "enable_hyperparameter_tuning": False,
    "register_trained_models": True,
    "store_training_artifacts": True,
}


# -----------------------------------------------------------------------------
# Backward-compatible notebook aliases
# -----------------------------------------------------------------------------

CANDIDATE_MODELS = MODEL_CONFIG["enabled_models"]
PRIMARY_MODEL_METRIC = MODEL_CONFIG["primary_metric"]
SECONDARY_MODEL_METRICS = MODEL_CONFIG["secondary_metrics"]
MODEL_RANDOM_SEED = MODEL_CONFIG["random_seed"]


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert CANDIDATE_MODELS == (
    "naive_last_value",
    "linear_regression",
    "random_forest",
)

assert PRIMARY_MODEL_METRIC == "wape"

assert SECONDARY_MODEL_METRICS == (
    "mae",
    "rmse",
    "mape",
    "smape",
    "bias",
)

assert MODEL_CONFIG["random_seed"] == 42

assert (
    MODEL_CONFIG["training_ratio"]
    + MODEL_CONFIG["validation_ratio"]
    + MODEL_CONFIG["test_ratio"]
    == 1.0
)

assert MODEL_CONFIG["shuffle_training"] is False
assert MODEL_CONFIG["register_trained_models"] is True
assert MODEL_CONFIG["store_training_artifacts"] is True


print("=" * 80)
print("MODEL RUNTIME PARAMETERS")
print("=" * 80)
print(f"Enabled models:       {CANDIDATE_MODELS}")
print(f"Primary metric:       {PRIMARY_MODEL_METRIC}")
print(f"Secondary metrics:    {SECONDARY_MODEL_METRICS}")
print(f"Random seed:          {MODEL_RANDOM_SEED}")
print(
    "Dataset split:        "
    f"{MODEL_CONFIG['training_ratio']:.2f}/"
    f"{MODEL_CONFIG['validation_ratio']:.2f}/"
    f"{MODEL_CONFIG['test_ratio']:.2f}"
)
print(
    "Shuffle training:     "
    f"{MODEL_CONFIG['shuffle_training']}"
)
print(
    "Register models:      "
    f"{MODEL_CONFIG['register_trained_models']}"
)
print("Model contract:       PASSED")
print("=" * 80)

## Section 06 — Capacity-Planning Parameters

These values are defaults for the public-data prototype.
Operational production values must be confirmed with warehouse
stakeholders before real-world deployment.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 06:
#     Capacity-Planning Runtime Parameters
#
# Release:
#     v3.0.0
# =============================================================================


# -----------------------------------------------------------------------------
# Workforce-capacity defaults
# -----------------------------------------------------------------------------
#
# These values provide runtime defaults for the public-data prototype.
# Production operational values must be validated with warehouse stakeholders.
#
# Capacity-gap calculations, staffing recommendations, and overtime decisions
# belong to the canonical src.workforce, src.planning, and src.overtime
# packages rather than this bootstrap notebook.
# -----------------------------------------------------------------------------

CAPACITY_CONFIG: dict[str, Any] = {
    "scheduled_hours": 10.0,
    "maximum_daily_hours": 10.0,
    "productivity_unit": "order_lines_per_associate_hour",
    "target_utilization": 0.85,
    "safety_buffer_ratio": 0.10,
    "forecast_confidence": 0.95,
}


# -----------------------------------------------------------------------------
# Overtime policy defaults
# -----------------------------------------------------------------------------

OVERTIME_POLICY: dict[str, Any] = {
    "voluntary_enabled": True,
    "mandatory_enabled": True,
    "allowed_day_types": (
        "OFF_DAY",
        "SATURDAY",
        "SUNDAY",
    ),
    "weekend_enabled": True,
    "holiday_adjustment_enabled": True,
    "minimum_shift_hours": 5.0,
    "maximum_shift_hours": 10.0,
}


# -----------------------------------------------------------------------------
# Service-level agreement defaults
# -----------------------------------------------------------------------------

SLA_CONFIG: dict[str, Any] = {
    "processing_commitment_hours": 48,
    "workload_unit": "order_lines",
    "include_current_oc_backlog": True,
    "include_projected_oc_backlog": True,
    "risk_levels": (
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL",
    ),
}


# -----------------------------------------------------------------------------
# Planning-horizon defaults
# -----------------------------------------------------------------------------

PLANNING_HORIZON_CONFIG: dict[str, Any] = {
    "next_day_days": 1,
    "weekly_days": 7,
    "monthly_days": 30,
    "quarterly_days": 90,
    "supported_horizon_days": (
        1,
        7,
        14,
        30,
        60,
        90,
    ),
}


# -----------------------------------------------------------------------------
# Backward-compatible notebook aliases
# -----------------------------------------------------------------------------

STANDARD_SHIFT_HOURS = CAPACITY_CONFIG["scheduled_hours"]
MAXIMUM_DAILY_HOURS = CAPACITY_CONFIG["maximum_daily_hours"]

MINIMUM_OVERTIME_HOURS = OVERTIME_POLICY["minimum_shift_hours"]
MAXIMUM_OVERTIME_HOURS = OVERTIME_POLICY["maximum_shift_hours"]

PROCESSING_COMMITMENT_HOURS = SLA_CONFIG[
    "processing_commitment_hours"
]


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert CAPACITY_CONFIG["scheduled_hours"] == 10.0
assert CAPACITY_CONFIG["maximum_daily_hours"] == 10.0

assert (
    CAPACITY_CONFIG["productivity_unit"]
    == "order_lines_per_associate_hour"
)

assert 0.0 < CAPACITY_CONFIG["target_utilization"] <= 1.0
assert 0.0 <= CAPACITY_CONFIG["safety_buffer_ratio"] < 1.0
assert 0.0 < CAPACITY_CONFIG["forecast_confidence"] <= 1.0

assert OVERTIME_POLICY["minimum_shift_hours"] == 5.0
assert OVERTIME_POLICY["maximum_shift_hours"] == 10.0

assert (
    OVERTIME_POLICY["minimum_shift_hours"]
    <= OVERTIME_POLICY["maximum_shift_hours"]
)

assert SLA_CONFIG["processing_commitment_hours"] == 48
assert SLA_CONFIG["workload_unit"] == "order_lines"

assert PLANNING_HORIZON_CONFIG["supported_horizon_days"] == (
    1,
    7,
    14,
    30,
    60,
    90,
)

assert (
    PLANNING_HORIZON_CONFIG["supported_horizon_days"]
    == SUPPORTED_FORECAST_HORIZON_DAYS
)


print("=" * 80)
print("CAPACITY-PLANNING RUNTIME PARAMETERS")
print("=" * 80)
print(
    f"Scheduled hours:       "
    f"{CAPACITY_CONFIG['scheduled_hours']}"
)
print(
    f"Maximum daily hours:   "
    f"{CAPACITY_CONFIG['maximum_daily_hours']}"
)
print(
    f"Productivity unit:     "
    f"{CAPACITY_CONFIG['productivity_unit']}"
)
print(
    f"Target utilization:    "
    f"{CAPACITY_CONFIG['target_utilization']}"
)
print(
    f"Safety buffer ratio:   "
    f"{CAPACITY_CONFIG['safety_buffer_ratio']}"
)
print(
    f"Minimum OT hours:      "
    f"{OVERTIME_POLICY['minimum_shift_hours']}"
)
print(
    f"Maximum OT hours:      "
    f"{OVERTIME_POLICY['maximum_shift_hours']}"
)
print(
    f"SLA commitment:        "
    f"{SLA_CONFIG['processing_commitment_hours']} hours"
)
print(
    f"Planning horizons:     "
    f"{PLANNING_HORIZON_CONFIG['supported_horizon_days']}"
)
print("Capacity contract:     PASSED")
print("=" * 80)

## Section 07 — AI-Assistant Parameters

Secrets and credentials must never be stored in this configuration.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 07:
#     AI-Assistant Runtime Parameters
#
# Release:
#     v3.0.0
# =============================================================================


AI_CONFIG: dict[str, Any] = {
    # -------------------------------------------------------------------------
    # Feature control
    # -------------------------------------------------------------------------
    "assistant_enabled": True,
    "human_review_required": True,

    # -------------------------------------------------------------------------
    # Context available to AI-assisted decision support
    # -------------------------------------------------------------------------
    "include_forecast_context": True,
    "include_capacity_context": True,
    "include_sla_context": True,
    "include_oc_backlog_context": True,
    "include_overtime_policy_context": True,
    "include_decision_explanation": True,

    # -------------------------------------------------------------------------
    # Canonical planning horizons
    # -------------------------------------------------------------------------
    "supported_horizons_days": (
        PLANNING_HORIZON_CONFIG["supported_horizon_days"]
    ),

    # -------------------------------------------------------------------------
    # Response controls
    # -------------------------------------------------------------------------
    "maximum_context_records": 30,
    "response_style": "operations_management",
    "require_evidence_summary": True,
    "require_confidence_statement": True,

    # -------------------------------------------------------------------------
    # Safety and governance
    # -------------------------------------------------------------------------
    "allow_autonomous_workforce_decisions": False,
    "allow_autonomous_overtime_scheduling": False,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert AI_CONFIG["assistant_enabled"] is True
assert AI_CONFIG["human_review_required"] is True

assert AI_CONFIG["supported_horizons_days"] == (
    1,
    7,
    14,
    30,
    60,
    90,
)

assert (
    AI_CONFIG["supported_horizons_days"]
    == PLANNING_HORIZON_CONFIG["supported_horizon_days"]
)

assert (
    AI_CONFIG["supported_horizons_days"]
    == SUPPORTED_FORECAST_HORIZON_DAYS
)

assert AI_CONFIG["maximum_context_records"] > 0

assert AI_CONFIG["require_evidence_summary"] is True
assert AI_CONFIG["require_confidence_statement"] is True

assert (
    AI_CONFIG["allow_autonomous_workforce_decisions"]
    is False
)

assert (
    AI_CONFIG["allow_autonomous_overtime_scheduling"]
    is False
)


# -----------------------------------------------------------------------------
# Security contract
# -----------------------------------------------------------------------------
#
# Provider credentials, API keys, tokens, passwords, and secrets must never
# be stored in this notebook configuration.
# -----------------------------------------------------------------------------

FORBIDDEN_SECRET_KEYS = {
    "api_key",
    "token",
    "password",
    "secret",
    "client_secret",
    "access_key",
}

assert not (
    FORBIDDEN_SECRET_KEYS
    & {key.lower() for key in AI_CONFIG}
)


print("=" * 80)
print("AI-ASSISTANT RUNTIME PARAMETERS")
print("=" * 80)
print(
    f"Assistant enabled:     "
    f"{AI_CONFIG['assistant_enabled']}"
)
print(
    f"Human review required: "
    f"{AI_CONFIG['human_review_required']}"
)
print(
    f"Planning horizons:     "
    f"{AI_CONFIG['supported_horizons_days']}"
)
print(
    f"Response style:        "
    f"{AI_CONFIG['response_style']}"
)
print(
    f"Evidence required:     "
    f"{AI_CONFIG['require_evidence_summary']}"
)
print(
    f"Confidence required:   "
    f"{AI_CONFIG['require_confidence_statement']}"
)
print(
    f"Autonomous workforce:  "
    f"{AI_CONFIG['allow_autonomous_workforce_decisions']}"
)
print(
    f"Autonomous OT:         "
    f"{AI_CONFIG['allow_autonomous_overtime_scheduling']}"
)
print("Secret config check:   PASSED")
print("AI governance contract: PASSED")
print("=" * 80)

## Section 08 — Shared Validation Utilities

Reusable validation functions for configuration dictionaries, numeric  
parameters, runtime horizons, and controlled option values.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 08:
#     Shared Bootstrap Validation Utilities
#
# Release:
#     v3.0.0
# =============================================================================

from collections.abc import Mapping, Sequence
from typing import Any


# -----------------------------------------------------------------------------
# Required-key validation
# -----------------------------------------------------------------------------

def validate_required_keys(
    *,
    config_name: str,
    config: Mapping[str, Any],
    required_keys: Sequence[str],
) -> None:
    """
    Validate that a configuration contains all required non-null keys.
    """

    if not isinstance(config, Mapping):
        raise TypeError(
            f"{config_name} must be a mapping, "
            f"received {type(config).__name__}."
        )

    missing_keys = [
        key
        for key in required_keys
        if key not in config or config[key] is None
    ]

    if missing_keys:
        raise ValueError(
            f"{config_name} is missing required keys: "
            f"{sorted(missing_keys)}"
        )


# -----------------------------------------------------------------------------
# Integer-range validation
# -----------------------------------------------------------------------------

def validate_integer_range(
    *,
    parameter_name: str,
    value: Any,
    minimum: int,
    maximum: int,
) -> int:
    """
    Validate that a value is an integer within an inclusive range.
    """

    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError(
            f"{parameter_name} must be an integer, "
            f"received {type(value).__name__}."
        )

    if minimum > maximum:
        raise ValueError(
            f"Invalid validation range for {parameter_name}: "
            f"minimum {minimum} exceeds maximum {maximum}."
        )

    if not minimum <= value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between "
            f"{minimum} and {maximum}; received {value}."
        )

    return value


# -----------------------------------------------------------------------------
# Numeric-range validation
# -----------------------------------------------------------------------------

def validate_numeric_range(
    *,
    parameter_name: str,
    value: Any,
    minimum: float,
    maximum: float,
) -> float:
    """
    Validate that a numeric value is within an inclusive range.
    """

    if (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
    ):
        raise TypeError(
            f"{parameter_name} must be numeric, "
            f"received {type(value).__name__}."
        )

    numeric_value = float(value)

    if minimum > maximum:
        raise ValueError(
            f"Invalid validation range for {parameter_name}: "
            f"minimum {minimum} exceeds maximum {maximum}."
        )

    if not minimum <= numeric_value <= maximum:
        raise ValueError(
            f"{parameter_name} must be between "
            f"{minimum} and {maximum}; "
            f"received {numeric_value}."
        )

    return numeric_value


# -----------------------------------------------------------------------------
# Non-empty-string validation
# -----------------------------------------------------------------------------

def validate_non_empty_string(
    *,
    parameter_name: str,
    value: Any,
) -> str:
    """
    Validate that a value is a non-empty string.
    """

    if not isinstance(value, str):
        raise TypeError(
            f"{parameter_name} must be a string, "
            f"received {type(value).__name__}."
        )

    normalized_value = value.strip()

    if not normalized_value:
        raise ValueError(
            f"{parameter_name} cannot be empty."
        )

    return normalized_value


# -----------------------------------------------------------------------------
# Controlled-string validation
# -----------------------------------------------------------------------------

def validate_string_choice(
    *,
    parameter_name: str,
    value: Any,
    allowed_values: Sequence[str],
) -> str:
    """
    Validate that a string belongs to a controlled set of values.
    """

    normalized_value = validate_non_empty_string(
        parameter_name=parameter_name,
        value=value,
    )

    allowed_set = set(allowed_values)

    if not allowed_set:
        raise ValueError(
            "No allowed values were configured for "
            f"{parameter_name}."
        )

    if normalized_value not in allowed_set:
        raise ValueError(
            f"{parameter_name} must be one of "
            f"{sorted(allowed_set)}; "
            f"received {normalized_value!r}."
        )

    return normalized_value


# -----------------------------------------------------------------------------
# Boolean validation
# -----------------------------------------------------------------------------

def validate_boolean(
    *,
    parameter_name: str,
    value: Any,
) -> bool:
    """
    Validate that a configuration value is Boolean.
    """

    if not isinstance(value, bool):
        raise TypeError(
            f"{parameter_name} must be Boolean, "
            f"received {type(value).__name__}."
        )

    return value


# -----------------------------------------------------------------------------
# Non-empty-sequence validation
# -----------------------------------------------------------------------------

def validate_non_empty_sequence(
    *,
    parameter_name: str,
    value: Any,
) -> Sequence[Any]:
    """
    Validate a non-empty list or tuple configuration value.
    """

    if (
        isinstance(value, (str, bytes))
        or not isinstance(value, Sequence)
    ):
        raise TypeError(
            f"{parameter_name} must be a list or tuple, "
            f"received {type(value).__name__}."
        )

    if len(value) == 0:
        raise ValueError(
            f"{parameter_name} cannot be empty."
        )

    return value


# -----------------------------------------------------------------------------
# Utility contract validation
# -----------------------------------------------------------------------------

validate_required_keys(
    config_name="PROJECT_CONFIG",
    config=PROJECT_CONFIG,
    required_keys=(
        "project_name",
        "project_key",
        "project_version",
        "environment",
    ),
)

assert (
    validate_integer_range(
        parameter_name="forecast_horizon",
        value=14,
        minimum=1,
        maximum=90,
    )
    == 14
)

assert (
    validate_numeric_range(
        parameter_name="confidence_level",
        value=0.95,
        minimum=0.0,
        maximum=1.0,
    )
    == 0.95
)

assert (
    validate_non_empty_string(
        parameter_name="project_name",
        value=PROJECT_NAME,
    )
    == PROJECT_NAME
)

assert (
    validate_string_choice(
        parameter_name="environment",
        value=ENVIRONMENT,
        allowed_values=(
            "development",
            "test",
            "staging",
            "production",
        ),
    )
    == ENVIRONMENT
)

assert (
    validate_boolean(
        parameter_name="human_review_required",
        value=AI_CONFIG["human_review_required"],
    )
    is True
)

assert tuple(
    validate_non_empty_sequence(
        parameter_name="supported_forecast_horizons",
        value=SUPPORTED_FORECAST_HORIZON_DAYS,
    )
) == SUPPORTED_FORECAST_HORIZON_DAYS


print("=" * 80)
print("SHARED BOOTSTRAP VALIDATION UTILITIES")
print("=" * 80)
print("Required-key validator:     PASSED")
print("Integer-range validator:    PASSED")
print("Numeric-range validator:    PASSED")
print("String validator:           PASSED")
print("Controlled-choice validator: PASSED")
print("Boolean validator:          PASSED")
print("Sequence validator:         PASSED")
print("Validation utilities:       PASSED")
print("=" * 80)

## Section 09 — Runtime Parameter Contract  

Resolve the active forecast horizon while preserving configured minimum,  
default, and maximum planning boundaries.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 09:
#     Runtime Parameter Contract
#
# Release:
#     v3.0.0
# =============================================================================


def resolve_forecast_horizon(
    requested_horizon: int | None = None,
) -> int:
    """
    Resolve and validate the active forecast horizon.

    When no runtime override is supplied, the configured default horizon is
    used. Runtime overrides must belong to the canonical set of supported
    forecast horizons.
    """

    horizon = (
        DEFAULT_FORECAST_HORIZON_DAYS
        if requested_horizon is None
        else requested_horizon
    )

    # Validate integer semantics first.
    validated_horizon = validate_integer_range(
        parameter_name="forecast_horizon_days",
        value=horizon,
        minimum=MINIMUM_FORECAST_HORIZON_DAYS,
        maximum=MAXIMUM_FORECAST_HORIZON_DAYS,
    )

    # Then enforce the actual platform-supported horizon contract.
    if validated_horizon not in SUPPORTED_FORECAST_HORIZON_DAYS:
        raise ValueError(
            "forecast_horizon_days must be one of "
            f"{SUPPORTED_FORECAST_HORIZON_DAYS}; "
            f"received {validated_horizon}."
        )

    return validated_horizon


# -----------------------------------------------------------------------------
# Active runtime parameters
# -----------------------------------------------------------------------------

ACTIVE_FORECAST_HORIZON_DAYS = resolve_forecast_horizon()


RUNTIME_CONFIG: dict[str, Any] = {
    "forecast_horizon_days": ACTIVE_FORECAST_HORIZON_DAYS,
    "forecast_frequency": FORECAST_CONFIG["frequency"],
    "forecast_target": FORECAST_CONFIG["target_column"],
    "forecast_date_column": FORECAST_CONFIG["date_column"],
    "environment": ENVIRONMENT,
    "platform_release": PLATFORM_RELEASE,
    "initialized_at_utc": PROJECT_INITIALIZED_AT_UTC,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert (
    ACTIVE_FORECAST_HORIZON_DAYS
    == DEFAULT_FORECAST_HORIZON_DAYS
)

assert (
    ACTIVE_FORECAST_HORIZON_DAYS
    in SUPPORTED_FORECAST_HORIZON_DAYS
)

assert resolve_forecast_horizon(1) == 1
assert resolve_forecast_horizon(7) == 7
assert resolve_forecast_horizon(14) == 14
assert resolve_forecast_horizon(30) == 30
assert resolve_forecast_horizon(60) == 60
assert resolve_forecast_horizon(90) == 90

assert (
    RUNTIME_CONFIG["forecast_target"]
    == "order_line_count"
)

assert (
    RUNTIME_CONFIG["forecast_frequency"]
    == "daily"
)

assert (
    RUNTIME_CONFIG["platform_release"]
    == PLATFORM_RELEASE
)


# -----------------------------------------------------------------------------
# Reject unsupported horizons
# -----------------------------------------------------------------------------

try:
    resolve_forecast_horizon(17)

except ValueError:
    pass

else:
    raise AssertionError(
        "Unsupported forecast horizon 17 must be rejected."
    )


print("=" * 80)
print("RUNTIME PARAMETER CONTRACT")
print("=" * 80)
print(
    f"Active forecast horizon: "
    f"{ACTIVE_FORECAST_HORIZON_DAYS} days"
)
print(
    f"Supported horizons:      "
    f"{SUPPORTED_FORECAST_HORIZON_DAYS}"
)
print(
    f"Forecast frequency:      "
    f"{RUNTIME_CONFIG['forecast_frequency']}"
)
print(
    f"Forecast target:         "
    f"{RUNTIME_CONFIG['forecast_target']}"
)
print(
    f"Environment:             "
    f"{RUNTIME_CONFIG['environment']}"
)
print(
    f"Platform release:        "
    f"{RUNTIME_CONFIG['platform_release']}"
)
print("Unsupported horizon test: PASSED")
print("Runtime contract:         PASSED")
print("=" * 80)

## Section 10 — Platform Configuration Validation  

Validate all centralized platform contracts before downstream notebooks  
consume the shared configuration.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 10:
#     Platform Configuration Validation
#
# Release:
#     v3.0.0
# =============================================================================


def validate_platform_configuration() -> None:
    """
    Validate all centralized platform configuration contracts.

    This validation protects the shared configuration consumed by downstream
    notebooks and verifies cross-domain consistency between forecasting,
    capacity planning, overtime, SLA, and AI-assistant configuration.
    """

    # =========================================================================
    # Required configuration keys
    # =========================================================================

    validate_required_keys(
        config_name="PROJECT_CONFIG",
        config=PROJECT_CONFIG,
        required_keys=(
            "project_name",
            "project_key",
            "project_version",
            "project_setup_version",
            "environment",
            "project_owner",
        ),
    )

    validate_required_keys(
        config_name="STORAGE_CONFIG",
        config=STORAGE_CONFIG,
        required_keys=(
            "project_root",
            "bronze_root",
            "silver_root",
            "gold_root",
            "metadata_root",
            "registry_root",
            "dataset_registry_path",
            "manifest_root",
            "validation_root",
            "pipeline_log_root",
            "model_root",
            "forecast_root",
            "decision_root",
            "report_root",
        ),
    )

    validate_required_keys(
        config_name="PIPELINE_CONFIG",
        config=PIPELINE_CONFIG,
        required_keys=(
            "pipeline_name",
            "pipeline_version",
            "write_mode",
            "parquet_compression",
            "enable_data_quality_checks",
            "save_execution_log",
            "fail_on_empty_dataset",
            "fail_on_row_count_mismatch",
            "fail_on_duplicate_business_keys",
        ),
    )

    validate_required_keys(
        config_name="FORECAST_CONFIG",
        config=FORECAST_CONFIG,
        required_keys=(
            "default_horizon_days",
            "frequency",
            "date_column",
            "target_column",
            "validation_horizon_days",
            "minimum_training_rows",
            "random_seed",
            "retrain_model",
            "save_model",
            "confidence_level",
        ),
    )

    validate_required_keys(
        config_name="MODEL_CONFIG",
        config=MODEL_CONFIG,
        required_keys=(
            "enabled_models",
            "primary_metric",
            "secondary_metrics",
            "random_seed",
            "training_ratio",
            "validation_ratio",
            "test_ratio",
            "shuffle_training",
            "register_trained_models",
        ),
    )

    validate_required_keys(
        config_name="CAPACITY_CONFIG",
        config=CAPACITY_CONFIG,
        required_keys=(
            "scheduled_hours",
            "maximum_daily_hours",
            "productivity_unit",
            "target_utilization",
            "safety_buffer_ratio",
        ),
    )

    validate_required_keys(
        config_name="OVERTIME_POLICY",
        config=OVERTIME_POLICY,
        required_keys=(
            "voluntary_enabled",
            "mandatory_enabled",
            "allowed_day_types",
            "weekend_enabled",
            "holiday_adjustment_enabled",
            "minimum_shift_hours",
            "maximum_shift_hours",
        ),
    )

    validate_required_keys(
        config_name="SLA_CONFIG",
        config=SLA_CONFIG,
        required_keys=(
            "processing_commitment_hours",
            "workload_unit",
            "include_current_oc_backlog",
            "include_projected_oc_backlog",
            "risk_levels",
        ),
    )

    validate_required_keys(
        config_name="PLANNING_HORIZON_CONFIG",
        config=PLANNING_HORIZON_CONFIG,
        required_keys=(
            "next_day_days",
            "weekly_days",
            "monthly_days",
            "quarterly_days",
            "supported_horizon_days",
        ),
    )

    validate_required_keys(
        config_name="AI_CONFIG",
        config=AI_CONFIG,
        required_keys=(
            "assistant_enabled",
            "human_review_required",
            "include_forecast_context",
            "include_capacity_context",
            "include_sla_context",
            "include_oc_backlog_context",
            "include_overtime_policy_context",
            "include_decision_explanation",
            "supported_horizons_days",
            "maximum_context_records",
            "response_style",
            "require_evidence_summary",
            "require_confidence_statement",
            "allow_autonomous_workforce_decisions",
            "allow_autonomous_overtime_scheduling",
        ),
    )

    # =========================================================================
    # Project contract
    # =========================================================================

    validate_non_empty_string(
        parameter_name="project_name",
        value=PROJECT_CONFIG["project_name"],
    )

    validate_non_empty_string(
        parameter_name="project_key",
        value=PROJECT_CONFIG["project_key"],
    )

    validate_string_choice(
        parameter_name="environment",
        value=PROJECT_CONFIG["environment"],
        allowed_values=(
            "development",
            "test",
            "staging",
            "production",
        ),
    )

    if f"v{PROJECT_CONFIG['project_version']}" != PLATFORM_RELEASE:
        raise ValueError(
            "PROJECT_CONFIG project_version must match PLATFORM_RELEASE."
        )

    if f"v{PROJECT_CONFIG['project_setup_version']}" != PLATFORM_RELEASE:
        raise ValueError(
            "PROJECT_CONFIG project_setup_version must match PLATFORM_RELEASE."
        )

    # =========================================================================
    # Storage contract
    # =========================================================================

    for storage_key in (
        "project_root",
        "bronze_root",
        "silver_root",
        "gold_root",
        "metadata_root",
        "registry_root",
        "dataset_registry_path",
        "manifest_root",
        "validation_root",
        "pipeline_log_root",
        "model_root",
        "forecast_root",
        "decision_root",
        "report_root",
    ):
        validate_non_empty_string(
            parameter_name=storage_key,
            value=STORAGE_CONFIG[storage_key],
        )

    # =========================================================================
    # Pipeline contract
    # =========================================================================

    validate_string_choice(
        parameter_name="pipeline_write_mode",
        value=PIPELINE_CONFIG["write_mode"],
        allowed_values=(
            "overwrite",
            "append",
            "error",
            "ignore",
        ),
    )

    validate_non_empty_string(
        parameter_name="parquet_compression",
        value=PIPELINE_CONFIG["parquet_compression"],
    )

    for boolean_key in (
        "enable_data_quality_checks",
        "save_execution_log",
        "fail_on_empty_dataset",
        "fail_on_row_count_mismatch",
        "fail_on_duplicate_business_keys",
    ):
        validate_boolean(
            parameter_name=boolean_key,
            value=PIPELINE_CONFIG[boolean_key],
        )

    # =========================================================================
    # Forecast contract
    # =========================================================================

    default_horizon = validate_integer_range(
        parameter_name="default_horizon_days",
        value=FORECAST_CONFIG["default_horizon_days"],
        minimum=MINIMUM_FORECAST_HORIZON_DAYS,
        maximum=MAXIMUM_FORECAST_HORIZON_DAYS,
    )

    if default_horizon not in SUPPORTED_FORECAST_HORIZON_DAYS:
        raise ValueError(
            "Default forecast horizon must belong to "
            f"{SUPPORTED_FORECAST_HORIZON_DAYS}; "
            f"received {default_horizon}."
        )

    validate_integer_range(
        parameter_name="validation_horizon_days",
        value=FORECAST_CONFIG["validation_horizon_days"],
        minimum=1,
        maximum=MAXIMUM_FORECAST_HORIZON_DAYS,
    )

    validate_integer_range(
        parameter_name="minimum_training_rows",
        value=FORECAST_CONFIG["minimum_training_rows"],
        minimum=1,
        maximum=10_000_000,
    )

    validate_integer_range(
        parameter_name="forecast_random_seed",
        value=FORECAST_CONFIG["random_seed"],
        minimum=0,
        maximum=2_147_483_647,
    )

    validate_numeric_range(
        parameter_name="confidence_level",
        value=FORECAST_CONFIG["confidence_level"],
        minimum=0.50,
        maximum=0.999,
    )

    validate_boolean(
        parameter_name="retrain_model",
        value=FORECAST_CONFIG["retrain_model"],
    )

    validate_boolean(
        parameter_name="save_model",
        value=FORECAST_CONFIG["save_model"],
    )

    # =========================================================================
    # Model contract
    # =========================================================================

    enabled_models = validate_non_empty_sequence(
        parameter_name="enabled_models",
        value=MODEL_CONFIG["enabled_models"],
    )

    for model_name in enabled_models:
        validate_non_empty_string(
            parameter_name="enabled_model",
            value=model_name,
        )

    validate_non_empty_string(
        parameter_name="primary_metric",
        value=MODEL_CONFIG["primary_metric"],
    )

    secondary_metrics = validate_non_empty_sequence(
        parameter_name="secondary_metrics",
        value=MODEL_CONFIG["secondary_metrics"],
    )

    for metric_name in secondary_metrics:
        validate_non_empty_string(
            parameter_name="secondary_metric",
            value=metric_name,
        )

    validate_integer_range(
        parameter_name="model_random_seed",
        value=MODEL_CONFIG["random_seed"],
        minimum=0,
        maximum=2_147_483_647,
    )

    training_ratio = validate_numeric_range(
        parameter_name="training_ratio",
        value=MODEL_CONFIG["training_ratio"],
        minimum=0.0,
        maximum=1.0,
    )

    validation_ratio = validate_numeric_range(
        parameter_name="validation_ratio",
        value=MODEL_CONFIG["validation_ratio"],
        minimum=0.0,
        maximum=1.0,
    )

    test_ratio = validate_numeric_range(
        parameter_name="test_ratio",
        value=MODEL_CONFIG["test_ratio"],
        minimum=0.0,
        maximum=1.0,
    )

    split_total = training_ratio + validation_ratio + test_ratio

    if abs(split_total - 1.0) > 1e-9:
        raise ValueError(
            "Model train/validation/test ratios must sum to 1.0; "
            f"received {split_total}."
        )

    validate_boolean(
        parameter_name="shuffle_training",
        value=MODEL_CONFIG["shuffle_training"],
    )

    validate_boolean(
        parameter_name="register_trained_models",
        value=MODEL_CONFIG["register_trained_models"],
    )

    # =========================================================================
    # Capacity-planning contract
    # =========================================================================

    scheduled_hours = validate_numeric_range(
        parameter_name="scheduled_hours",
        value=CAPACITY_CONFIG["scheduled_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    maximum_daily_hours = validate_numeric_range(
        parameter_name="maximum_daily_hours",
        value=CAPACITY_CONFIG["maximum_daily_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    if scheduled_hours > maximum_daily_hours:
        raise ValueError(
            "Standard shift hours cannot exceed maximum daily hours."
        )

    validate_non_empty_string(
        parameter_name="productivity_unit",
        value=CAPACITY_CONFIG["productivity_unit"],
    )

    validate_numeric_range(
        parameter_name="target_utilization",
        value=CAPACITY_CONFIG["target_utilization"],
        minimum=0.0,
        maximum=1.0,
    )

    validate_numeric_range(
        parameter_name="safety_buffer_ratio",
        value=CAPACITY_CONFIG["safety_buffer_ratio"],
        minimum=0.0,
        maximum=1.0,
    )

    # =========================================================================
    # Overtime contract
    # =========================================================================

    validate_boolean(
        parameter_name="voluntary_overtime_enabled",
        value=OVERTIME_POLICY["voluntary_enabled"],
    )

    validate_boolean(
        parameter_name="mandatory_overtime_enabled",
        value=OVERTIME_POLICY["mandatory_enabled"],
    )

    validate_boolean(
        parameter_name="weekend_enabled",
        value=OVERTIME_POLICY["weekend_enabled"],
    )

    validate_boolean(
        parameter_name="holiday_adjustment_enabled",
        value=OVERTIME_POLICY["holiday_adjustment_enabled"],
    )

    allowed_day_types = validate_non_empty_sequence(
        parameter_name="allowed_overtime_day_types",
        value=OVERTIME_POLICY["allowed_day_types"],
    )

    for day_type in allowed_day_types:
        validate_non_empty_string(
            parameter_name="overtime_day_type",
            value=day_type,
        )

    minimum_overtime_hours = validate_numeric_range(
        parameter_name="minimum_overtime_shift_hours",
        value=OVERTIME_POLICY["minimum_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    maximum_overtime_hours = validate_numeric_range(
        parameter_name="maximum_overtime_shift_hours",
        value=OVERTIME_POLICY["maximum_shift_hours"],
        minimum=1.0,
        maximum=24.0,
    )

    if minimum_overtime_hours > maximum_overtime_hours:
        raise ValueError(
            "Minimum overtime shift hours cannot exceed maximum "
            "overtime shift hours."
        )

    if maximum_overtime_hours > maximum_daily_hours:
        raise ValueError(
            "Maximum overtime shift hours cannot exceed configured "
            "maximum daily hours."
        )

    # =========================================================================
    # SLA contract
    # =========================================================================

    validate_integer_range(
        parameter_name="processing_commitment_hours",
        value=SLA_CONFIG["processing_commitment_hours"],
        minimum=1,
        maximum=720,
    )

    validate_non_empty_string(
        parameter_name="sla_workload_unit",
        value=SLA_CONFIG["workload_unit"],
    )

    validate_boolean(
        parameter_name="include_current_oc_backlog",
        value=SLA_CONFIG["include_current_oc_backlog"],
    )

    validate_boolean(
        parameter_name="include_projected_oc_backlog",
        value=SLA_CONFIG["include_projected_oc_backlog"],
    )

    risk_levels = validate_non_empty_sequence(
        parameter_name="sla_risk_levels",
        value=SLA_CONFIG["risk_levels"],
    )

    for risk_level in risk_levels:
        validate_non_empty_string(
            parameter_name="sla_risk_level",
            value=risk_level,
        )

# =========================================================================
# Planning-horizon contract
# =========================================================================

planning_horizons = tuple(
    validate_non_empty_sequence(
        parameter_name="supported_planning_horizons",
        value=PLANNING_HORIZON_CONFIG[
            "supported_horizon_days"
        ],
    )
)

assert planning_horizons

for horizon in planning_horizons:
    validate_integer_range(
        parameter_name="planning_horizon_days",
        value=horizon,
        minimum=MINIMUM_FORECAST_HORIZON_DAYS,
        maximum=MAXIMUM_FORECAST_HORIZON_DAYS,
    )

if (
    planning_horizons
    != SUPPORTED_FORECAST_HORIZON_DAYS
):
    raise ValueError(
        "Planning horizons must match the canonical "
        "forecast horizons. "
        f"Forecast={SUPPORTED_FORECAST_HORIZON_DAYS}; "
        f"Planning={planning_horizons}."
    )

if (
    PLANNING_HORIZON_CONFIG["next_day_days"]
    != 1
):
    raise ValueError(
        "Next-day planning horizon must equal 1 day."
    )

if (
    PLANNING_HORIZON_CONFIG["weekly_days"]
    != 7
):
    raise ValueError(
        "Weekly planning horizon must equal 7 days."
    )

if (
    PLANNING_HORIZON_CONFIG["monthly_days"]
    != 30
):
    raise ValueError(
        "Monthly planning horizon must equal 30 days."
    )

if (
    PLANNING_HORIZON_CONFIG["quarterly_days"]
    != 90
):
    raise ValueError(
        "Quarterly planning horizon must equal 90 days."
    )

    # =========================================================================
    # AI-assistant governance contract
    # =========================================================================

    validate_boolean(
        parameter_name="assistant_enabled",
        value=AI_CONFIG["assistant_enabled"],
    )

    validate_boolean(
        parameter_name="human_review_required",
        value=AI_CONFIG["human_review_required"],
    )

    for context_key in (
        "include_forecast_context",
        "include_capacity_context",
        "include_sla_context",
        "include_oc_backlog_context",
        "include_overtime_policy_context",
        "include_decision_explanation",
        "require_evidence_summary",
        "require_confidence_statement",
    ):
        validate_boolean(
            parameter_name=context_key,
            value=AI_CONFIG[context_key],
        )

    validate_integer_range(
        parameter_name="maximum_context_records",
        value=AI_CONFIG["maximum_context_records"],
        minimum=1,
        maximum=10_000,
    )

    validate_non_empty_string(
        parameter_name="response_style",
        value=AI_CONFIG["response_style"],
    )

    ai_supported_horizons = tuple(
        validate_non_empty_sequence(
            parameter_name="ai_supported_horizons_days",
            value=AI_CONFIG["supported_horizons_days"],
        )
    )

    if ai_supported_horizons != SUPPORTED_FORECAST_HORIZON_DAYS:
        raise ValueError(
            "AI-supported horizons must match the canonical forecast "
            "horizons."
        )

    if not AI_CONFIG["human_review_required"]:
        raise ValueError(
            "Human review must remain required for workforce decisions."
        )

    if AI_CONFIG["allow_autonomous_workforce_decisions"]:
        raise ValueError(
            "Autonomous workforce decisions are prohibited."
        )

    if AI_CONFIG["allow_autonomous_overtime_scheduling"]:
        raise ValueError(
            "Autonomous overtime scheduling is prohibited."
        )

    # =========================================================================
    # Runtime alignment
    # =========================================================================

    if (
        RUNTIME_CONFIG["forecast_horizon_days"]
        not in SUPPORTED_FORECAST_HORIZON_DAYS
    ):
        raise ValueError(
            "Active runtime forecast horizon is not supported."
        )

    if (
        RUNTIME_CONFIG["platform_release"]
        != PLATFORM_RELEASE
    ):
        raise ValueError(
            "Runtime platform release does not match PLATFORM_RELEASE."
        )


# =============================================================================
# Execute centralized configuration validation
# =============================================================================

validate_platform_configuration()

CONFIGURATION_STATUS = "PASSED"


print("=" * 80)
print("PLATFORM CONFIGURATION VALIDATION")
print("=" * 80)
print(f"Project contract:          {CONFIGURATION_STATUS}")
print(f"Storage contract:          {CONFIGURATION_STATUS}")
print(f"Pipeline contract:         {CONFIGURATION_STATUS}")
print(f"Forecast contract:         {CONFIGURATION_STATUS}")
print(f"Model contract:            {CONFIGURATION_STATUS}")
print(f"Capacity contract:         {CONFIGURATION_STATUS}")
print(f"Overtime contract:         {CONFIGURATION_STATUS}")
print(f"SLA contract:              {CONFIGURATION_STATUS}")
print(f"Planning horizon contract: {CONFIGURATION_STATUS}")
print(f"AI governance contract:    {CONFIGURATION_STATUS}")
print(f"Runtime alignment:         {CONFIGURATION_STATUS}")
print("-" * 80)
print(f"Configuration status:      {CONFIGURATION_STATUS}")
print("=" * 80)

## Section 11 — Storage Access Validation  

Verify that required persistent S3 locations are accessible before a  
downstream pipeline begins execution.

In [0]:
# =============================================================================
# AI Workforce Capacity Planning Platform
# Implementation 28 — Enterprise Release Engineering & Platform Hardening
#
# Notebook:
#     00_project_setup
#
# Section 11:
#     Storage Access Validation
#
# Release:
#     v3.0.0
# =============================================================================


def validate_storage_contract(
    paths: Mapping[str, str],
) -> dict[str, str]:
    """
    Validate canonical storage configuration.

    The bootstrap performs:
    1. structural validation for every configured storage path,
    2. one lightweight remote-access probe against the canonical project root.

    This avoids repeated S3 round-trips during notebook startup.
    """

    if not isinstance(paths, Mapping):
        raise TypeError(
            "Storage paths must be provided as a mapping."
        )

    if not paths:
        raise ValueError(
            "No storage paths were supplied for validation."
        )

    results: dict[str, str] = {}

    # -------------------------------------------------------------------------
    # Structural validation
    # -------------------------------------------------------------------------

    for path_name, path_value in paths.items():
        path = validate_non_empty_string(
            parameter_name=f"storage_path[{path_name}]",
            value=path_value,
        )

        if not (
            path == PROJECT_ROOT
            or path.startswith(f"{PROJECT_ROOT}/")
        ):
            raise ValueError(
                f"Storage path {path_name!r} is outside the canonical "
                f"project root.\n"
                f"Project root: {PROJECT_ROOT}\n"
                f"Path:         {path}"
            )

        results[path_name] = "CONFIGURED"

    # -------------------------------------------------------------------------
    # Single remote-access probe
    # -------------------------------------------------------------------------
    #
    # One listing is sufficient to confirm that Databricks can resolve the
    # canonical S3 project root. We intentionally avoid probing every nested
    # prefix during bootstrap.
    # -------------------------------------------------------------------------

    try:
        dbutils.fs.ls(PROJECT_ROOT)

    except Exception as exc:
        raise RuntimeError(
            "Canonical project storage root is not accessible.\n"
            f"Path: {PROJECT_ROOT}\n"
            f"Error: {type(exc).__name__}: {exc}"
        ) from exc

    results["project_root"] = "ACCESSIBLE"

    return results


# -----------------------------------------------------------------------------
# Canonical persistent S3 locations
# -----------------------------------------------------------------------------

STORAGE_VALIDATION_PATHS: dict[str, str] = {
    "project_root": PROJECT_ROOT,
    "landing_root": LANDING_ROOT,
    "bronze_root": BRONZE_ROOT,
    "silver_root": SILVER_ROOT,
    "gold_root": GOLD_ROOT,
    "metadata_root": METADATA_ROOT,
    "registry_root": REGISTRY_ROOT,
    "dataset_registry_path": DATASET_REGISTRY_PATH,
    "manifest_root": MANIFEST_ROOT,
    "validation_root": VALIDATION_ROOT,
    "pipeline_log_root": PIPELINE_LOG_ROOT,
    "model_root": MODEL_ROOT,
    "forecast_root": FORECAST_ROOT,
    "decision_root": DECISION_ROOT,
    "report_root": REPORT_ROOT,
}


# -----------------------------------------------------------------------------
# Contract validation
# -----------------------------------------------------------------------------

assert STORAGE_VALIDATION_PATHS["project_root"] == S3_ROOT

assert all(
    path == PROJECT_ROOT
    or path.startswith(f"{PROJECT_ROOT}/")
    for path in STORAGE_VALIDATION_PATHS.values()
)


# -----------------------------------------------------------------------------
# Execute storage validation
# -----------------------------------------------------------------------------

STORAGE_VALIDATION_RESULTS = validate_storage_contract(
    STORAGE_VALIDATION_PATHS
)

assert (
    STORAGE_VALIDATION_RESULTS["project_root"]
    == "ACCESSIBLE"
)

STORAGE_STATUS = "PASSED"
STORAGE_CONNECTION_OK = True
RUNTIME_STATUS = "READY"


print("=" * 80)
print("STORAGE ACCESS VALIDATION")
print("=" * 80)
print(f"Project root:        {PROJECT_ROOT}")
print(
    f"Configured locations: "
    f"{len(STORAGE_VALIDATION_PATHS)}"
)
print("Remote probes:       1")
print("Project root access: ACCESSIBLE")
print(f"Storage status:      {STORAGE_STATUS}")
print(f"Runtime status:      {RUNTIME_STATUS}")
print("=" * 80)

## Section 12 — Execution Summary  

Publish the validated platform bootstrap status for operators and  
downstream notebooks.

In [0]:
# =============================================================================
# Execution Summary
# =============================================================================

SETUP_EXECUTION_SUMMARY: dict[str, Any] = {
    "project_name": PROJECT_NAME,
    "project_key": PROJECT_KEY,
    "platform_release": PLATFORM_RELEASE,
    "project_version": PROJECT_VERSION,
    "project_setup_version": PROJECT_SETUP_VERSION,
    "environment": ENVIRONMENT,
    "canonical_namespace": "src.*",
    "project_root": PROJECT_ROOT,
    "active_forecast_horizon_days": ACTIVE_FORECAST_HORIZON_DAYS,
    "configuration_status": CONFIGURATION_STATUS,
    "storage_status": STORAGE_STATUS,
    "runtime_status": RUNTIME_STATUS,
    "initialized_at_utc": PROJECT_INITIALIZED_AT_UTC.isoformat(),
}

# -----------------------------------------------------------------------------
# Final execution-summary contract validation
# -----------------------------------------------------------------------------

assert (
    SETUP_EXECUTION_SUMMARY["platform_release"]
    == PLATFORM_RELEASE
)

assert (
    f"v{SETUP_EXECUTION_SUMMARY['project_version']}"
    == PLATFORM_RELEASE
)

assert (
    f"v{SETUP_EXECUTION_SUMMARY['project_setup_version']}"
    == PLATFORM_RELEASE
)

assert (
    SETUP_EXECUTION_SUMMARY["canonical_namespace"]
    == "src.*"
)

assert (
    SETUP_EXECUTION_SUMMARY["configuration_status"]
    == "PASSED"
)

assert (
    SETUP_EXECUTION_SUMMARY["storage_status"]
    == "PASSED"
)

assert (
    SETUP_EXECUTION_SUMMARY["runtime_status"]
    == "READY"
)

print("=" * 80)
print("AI WORKFORCE CAPACITY PLANNING PLATFORM")
print("PROJECT SETUP EXECUTION SUMMARY")
print("=" * 80)
print(f"Platform release       : {PLATFORM_RELEASE}")
print(f"Project name           : {PROJECT_NAME}")
print(f"Project key            : {PROJECT_KEY}")
print(f"Project version        : {PROJECT_VERSION}")
print(f"Setup version          : {PROJECT_SETUP_VERSION}")
print(f"Environment            : {ENVIRONMENT}")
print(f"Canonical namespace    : src.*")
print(f"Project root           : {PROJECT_ROOT}")
print(f"Forecast horizon       : {ACTIVE_FORECAST_HORIZON_DAYS} day(s)")
print(f"Configuration status   : {CONFIGURATION_STATUS}")
print(f"Storage status         : {STORAGE_STATUS}")
print(f"Runtime status         : {RUNTIME_STATUS}")
print("=" * 80)
print("PROJECT SETUP: PASSED")
print("=" * 80)